# Lab 4: Compress — Reducing Token Count\n\n**Concept:** Compression reduces the size of the context before sending it to the LLM. This can be rule-based (regex sentence dropping), statistical (TF-IDF filtering), or neural (LLM summarization). The trade-off is always token savings vs. information loss.\n\n## Exercises\n- 4a. Measure compression ratio and accuracy trade-off at different keyword thresholds (from `naive_regex`).\n- 4b. Use TF-IDF as a lossy compressor: keep only sentences above a similarity threshold (from `semantic_chunk`).\n- 4c. Use an LLM to compress a document at multiple compression levels and check fact preservation (from `llmlingua` & `pruning_dashboard`).\n

## Setup\n\n```bash\npip install openai scikit-learn numpy\nexport OPENROUTER_API_KEY='sk-or-v1-...'\n```

In [ ]:
import os, re, json, pickle, glob\nimport numpy as np\nfrom openai import OpenAI\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\n\nclient = OpenAI(api_key=os.getenv('OPENROUTER_API_KEY'), base_url='https://openrouter.ai/api/v1')

## Exercises

In [ ]:
import os, re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

import sys
sys.path.insert(0, os.path.abspath(".."))
try:
    from naive_regex_pruning.naive_regex import _naive_regex_prune
except ImportError:
    _naive_regex_prune = None

client = OpenAI(api_key=os.getenv("OPENROUTER_API_KEY"), base_url="https://openrouter.ai/api/v1")

document = """The company reported strong Q3 earnings. Revenue grew by 15% year over year to $45.2 million.
The expansion of the Frankfurt server farm cost $14.73 million. This was partially offset by a $2.1 million
green energy tax rebate. Employee headcount remained stable at 1,200. Management raised full-year guidance
citing strong demand in European markets."""
sentences = re.split(r'(?<=[.!?])\s+', document)
question = "What offset the Frankfurt data center costs?"

# === 4a: Keyword threshold compression ===
keywords = {w.lower() for w in re.findall(r'\b[a-zA-Z]{3,}\b', question)}
scored = [(sum(1 for k in keywords if k in s.lower()), s) for s in sentences]
scored.sort(key=lambda x: -x[0])

print("=== 4a: Threshold Compression ===")
for threshold in [0, 1, 2]:
    kept = [s for sc, s in scored if sc >= threshold] or sentences[:1]
    ratio = len(' '.join(kept).split()) / len(document.split()) * 100
    has_capex = "14.73" in ' '.join(kept)
    has_rebate = "2.1" in ' '.join(kept)
    print(f"  Threshold>={threshold}: {ratio:.0f}% | CAPEX={has_capex} | Rebate={has_rebate} | '{kept[0][:50]}...'")

# === 4b: TF-IDF similarity compression ===
vect = TfidfVectorizer(ngram_range=(1,2))
vect.fit([question] + sentences)
scores = cosine_similarity(vect.transform([question]), vect.transform(sentences)).flatten()

print("\n=== 4b: TF-IDF Similarity Compression ===")
for thresh in [0.0, 0.1, 0.3]:
    kept = [s for s, sc in zip(sentences, scores) if sc >= thresh]
    if not kept:
        kept = sentences[:1]
    ratio = len(' '.join(kept).split()) / len(document.split()) * 100
    print(f"  Sim>={thresh}: {ratio:.0f}% | {len(kept)} sentences kept | first: '{kept[0][:50]}...'")

# === 4c: LLM multi-level compression ===
print("\n=== 4c: LLM Compression Levels ===")
for instr in [
    "Summarize in one sentence keeping all numbers.",
    "Summarize in 3 bullet points.",
    "Extract only the named entities and numbers as a JSON object."
]:
    resp = client.chat.completions.create(
        model="meta-llama/llama-3.3-70b-instruct:free",
        messages=[{"role": "user", "content": f"{instr}\n\n{document}"}],
        temperature=0,
    )
    compressed = resp.choices[0].message.content
    ratio = len(compressed.split()) / len(document.split()) * 100
    has_capex = "14.73" in compressed
    has_rebate = "2.1" in compressed
    print(f"  '{instr[:40]}...' -> {ratio:.0f}% | CAPEX={has_capex} | Rebate={has_rebate}")

## Summary\n\nCompression reduces the size of the context before sending it to the LLM. This can be rule-based (regex sentence dropping), statistical (TF-IDF filtering), or neural (LLM summarization). The trade-off is always token savings vs. information loss.